# 03 Evaluate Results

Evaluate derivative MSE, RK4 rollout error, vector fields, and the learned Lyapunov function.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch

# Let the notebook import the local package whether Jupyter starts in the
# repository root or inside the notebooks/ directory.
REPO_ROOT = Path.cwd() if (Path.cwd() / "stable_icnn_physics").exists() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


from stable_icnn_physics import BaselineDynamicsMLP, DampedPendulum, MassSpringDamper
from stable_icnn_physics.data import dataset_path, load_dataset, tensor_dataset
from stable_icnn_physics.eval import lyapunov_decrease_values, rollout_error, rollout_model, rollout_system
from stable_icnn_physics.models import build_stable_model
from stable_icnn_physics.plotting import plot_lyapunov_contours, plot_rollout_error, plot_vector_field
from stable_icnn_physics.train import evaluate_derivative_mse

CACHE_DIR = REPO_ROOT / "data/cache"
OUTPUT_DIR = REPO_ROOT / "outputs"
SEED = 0
SYSTEM_NAME = "pendulum1"
N_TEST = 1000

system = DampedPendulum(n_links=1, friction=0.3, gravity=9.81)
# system = MassSpringDamper(mass=1.0, damping=0.3, stiffness=1.0)

HIDDEN = 100
DEPTH = 2
LYAPUNOV_HIDDEN = 60
LYAPUNOV_EPS = 0.01
ALPHA = 1e-3
ROLLOUT_STEPS = 300
DT = 0.01
N_ROLLOUTS = 64

In [ ]:
x_test, y_test = load_dataset(dataset_path(CACHE_DIR, SYSTEM_NAME, "test", N_TEST, SEED))
test_ds = tensor_dataset(x_test, y_test)
dim = x_test.shape[1]

stable_model = build_stable_model(dim, hidden=HIDDEN, depth=DEPTH, lyapunov_hidden=LYAPUNOV_HIDDEN, lyapunov_eps=LYAPUNOV_EPS, alpha=ALPHA)
baseline_model = BaselineDynamicsMLP(dim=dim, hidden=HIDDEN, depth=DEPTH)

stable_model.load_state_dict(torch.load(OUTPUT_DIR / f"{SYSTEM_NAME}_stable.pt", map_location="cpu")["model_state"])
baseline_model.load_state_dict(torch.load(OUTPUT_DIR / f"{SYSTEM_NAME}_baseline.pt", map_location="cpu")["model_state"])

print("stable derivative MSE:", evaluate_derivative_mse(stable_model, test_ds))
print("baseline derivative MSE:", evaluate_derivative_mse(baseline_model, test_ds))

In [ ]:
x0 = system.sample_states(N_ROLLOUTS, split="test", seed=SEED + 123)
true_traj = rollout_system(system, x0, steps=ROLLOUT_STEPS, dt=DT)
stable_traj = rollout_model(stable_model, x0, steps=ROLLOUT_STEPS, dt=DT, wrap_fn=system.wrap_state)
baseline_traj = rollout_model(baseline_model, x0, steps=ROLLOUT_STEPS, dt=DT, wrap_fn=system.wrap_state)

stable_error = rollout_error(system, true_traj, stable_traj)
baseline_error = rollout_error(system, true_traj, baseline_traj)
print("final stable mean error:", stable_error[-1].mean())
print("final baseline mean error:", baseline_error[-1].mean())

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
plot_rollout_error({"stable": stable_error, "baseline": baseline_error}, ax=ax)
plt.tight_layout()

In [ ]:
decrease = lyapunov_decrease_values(stable_model, x_test[:512])
print("max gradV*f + alpha*V:", decrease.max())
print("fraction <= 1e-5:", np.mean(decrease <= 1e-5))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_vector_field(system, model=None, ax=axes[0], title="True dynamics")
plot_vector_field(system, model=stable_model, ax=axes[1], title="Stable model")
plot_lyapunov_contours(stable_model, ax=axes[2], title="Learned V")
plt.tight_layout()